In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
df = spark.read.option("multiLine", "true").json("/Volumes/ipl/s1_raw/ipl_raw_data_json/*.json")

In [0]:
# Window for ranking within each season ordered by match_date where match_number is null


df = df.withColumn("match_number", col("info.event.match_number").cast("int"))\
        .withColumn("match_date",col("info.dates")[0].cast("date"))\
        .withColumn("season",col("info.season"))
w = Window.partitionBy("season", col("match_number").isNull()).orderBy("match_date")

df_final = df\
        .withColumn(
            "match_number",
            when(
                col("match_number").isNull(),
                (lit(99) * 10 + rank().over(w))
            ).otherwise(col("match_number"))
        )\
        .withColumn(
            "match_id", 
            concat(date_format("match_date", "yyyyMMdd"), lit("_"),lpad(col("match_number"), 3, "0"))
        )

display(df_final)

In [0]:
df_final = df_final.withColumn("team1",col("innings")[0].team)\
                    .withColumn("team2",
                                        when(
                                            col("info.outcome.result") != "no result",
                                            col("innings")[1].team
                                            )
                                        .otherwise(
                                            when(
                                                col("info.teams")[0] != col("innings")[0].team,
                                                col("info.teams")[0]
                                            ).otherwise(col("info.teams")[1])
                                        )
                                )

display(df_final)



In [0]:
display(df)

In [0]:
# Write the DataFrame to a table in the same schema
# Enable column mapping to allow spaces and special characters in column names
df_final.write.mode("overwrite").option("overwriteSchema", "true").option("mergeSchema", "true").option("columnMapping.mode", "name").saveAsTable("ipl.s1_raw.ipl_data")